# 🚀 Salting in Spark (Handling Data Skew Deep Dive)

---

# 1️⃣ What is Data Skew?

## 📌 Definition

> Data skew occurs when some keys have significantly more data than others.

---

## 🔹 Example

```
Key Distribution:

A → 90% of data
B → 5%
C → 5%
```

---

## 🔹 Problem

During operations like:

```python
df.groupBy("key").count()
```

or

```python
df1.join(df2, "key")
```

👉 All records with key "A" go to ONE partition

---

## 🔹 Impact

- One task becomes very slow  
- Other tasks finish early  
- Poor parallelism  
- Job slowdown  

---

# 2️⃣ What is Salting?

## 📌 Definition

> Salting is a technique to distribute skewed data evenly across partitions by adding a random suffix (salt) to keys.

---

## 🔹 Idea

Instead of:

```
A → One partition
```

We convert:

```
A → A_0, A_1, A_2, A_3, A_4
```

Now data spreads across multiple partitions.

---

# 3️⃣ How Salting Works (Step-by-Step)

---

## 🔹 Step 1: Add Salt to Skewed Dataset

```python
from pyspark.sql.functions import rand, floor

df1_salted = df1.withColumn("salt", floor(rand() * 5))
```

Now:

```
A → A_0, A_1, A_2, A_3, A_4
```

---

## 🔹 Step 2: Expand Small Dataset

```python
from pyspark.sql.functions import explode, array

df2_salted = df2.withColumn("salt", explode(array([0,1,2,3,4])))
```

Now small dataset is replicated:

```
A → A_0, A_1, A_2, A_3, A_4
```

---

## 🔹 Step 3: Join Using Key + Salt

```python
df_joined = df1_salted.join(df2_salted, ["key", "salt"])
```

---

## 🔹 What Happens Internally?

Before salting:

```
Partition 1 → All "A"
```

After salting:

```
Partition 1 → A_0
Partition 2 → A_1
Partition 3 → A_2
Partition 4 → A_3
Partition 5 → A_4
```

👉 Load distributed evenly

---

# 4️⃣ Hands-On Example

---

## 🔹 Create Skewed Data

```python
data = [("A", i) for i in range(1000)] + [("B", i) for i in range(10)]
df1 = spark.createDataFrame(data, ["key", "value"])

df2 = spark.createDataFrame([("A", "X"), ("B", "Y")], ["key", "desc"])
```

---

## 🔹 Normal Join (Skew Issue)

```python
df1.join(df2, "key").show()
```

👉 One partition overloaded

---

## 🔹 Apply Salting

```python
from pyspark.sql.functions import rand, floor, explode, array

# Add salt to large dataset
df1_salted = df1.withColumn("salt", floor(rand() * 5))

# Expand small dataset
df2_salted = df2.withColumn("salt", explode(array([0,1,2,3,4])))

# Join
df_result = df1_salted.join(df2_salted, ["key", "salt"])
```

---

## 🔹 Result

- Skew reduced  
- Better parallelism  
- Faster execution  

---

# 5️⃣ When to Use Salting?

---

## 🔥 Use Salting When:

- Data skew is present  
- Large joins are slow  
- One task takes much longer  
- Broadcast join is not possible  

---

## ❌ Avoid When:

- Data is evenly distributed  
- Small dataset (use broadcast instead)  

---

# 6️⃣ Trade-Offs

---

## 🔹 Pros

- Fixes skew  
- Improves parallelism  
- Reduces straggler tasks  

---

## 🔹 Cons

- Data duplication  
- Increased computation  
- More complex logic  

---

# 7️⃣ Alternative Solutions

---

## 🔹 1. Broadcast Join

```python
df1.join(broadcast(df2), "key")
```

---

## 🔹 2. AQE (Automatic Skew Handling)

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

## 🔹 3. Repartition

```python
df.repartition("key")
```

---

# 8️⃣ Interview-Level Questions

---

## ❓ What is salting?

👉 Technique to handle skew by adding random suffix to keys.

---

## ❓ Why is salting needed?

👉 To distribute skewed data across partitions.

---

## ❓ How does salting work?

👉 Split skewed key into multiple sub-keys and distribute load.

---

## ❓ When should you use salting?

👉 When skew exists and broadcast is not possible.

---

## ❓ Drawbacks of salting?

👉 Data duplication and extra computation.

---

# 🎯 Interview Answer (Best Version)

Salting is a technique used to handle data skew in Spark by adding a random suffix to skewed keys, thereby distributing the data across multiple partitions and improving parallelism during operations like joins and aggregations.

---

# 🚀 Final Summary

```
Skew → One partition overloaded
Salting → Split key into multiple parts
Result → Balanced workload
```

---

# 🔥 Golden Rule

👉 If one task is slow → Check skew → Apply salting or AQE  

# Hands On


In [0]:
from pyspark.sql.functions import rand, floor
floor(rand() * 5)

In [0]:
data = [("A", i) for i in range(1000)] + [("B", i) for i in range(10)]
df1 = spark.createDataFrame(data, ["key", "value"])

df2 = spark.createDataFrame([("A", "X"), ("B", "Y")], ["key", "desc"])

In [0]:
df1.display()
df2.display()

In [0]:
df1.join(df2, "key").show()

In [0]:
from pyspark.sql.functions import rand, floor, explode, array
print(floor(rand() * 5))

In [0]:
from pyspark.sql.functions import rand, floor, explode, array

# Add salt to large dataset
df1_salted = df1.withColumn("salt", floor(rand() * 5))

# Expand small dataset
from pyspark.sql.functions import lit
# Expand small dataset
df2_salted = df2.withColumn("salt", explode(array([lit(0), lit(1), lit(2), lit(3), lit(4)])))


In [0]:
df1_salted.display()
df2_salted.display()


In [0]:
# Join
df_result = df1_salted.join(df2_salted, ["key", "salt"])
df_result.display()

# 🚀 Spark Skew Debugging + Optimization + Mock Interview

---

# 1️⃣ Spark UI – Detecting Data Skew (Live Approach 🔥)

---

## 🔹 Step-by-Step

### Step 1: Run a Skewed Operation

```python
df.groupBy("key").count().show()
```

---

### Step 2: Open Spark UI

- Click → "View Spark UI"
- Go to → **Stages Tab**

---

### Step 3: Identify Slow Stage

Look for:

- Long execution time
- Shuffle-heavy stage

---

### Step 4: Open Stage → Tasks Tab

Now observe:

---

## 🔥 Signs of Skew

### ❗ Uneven Task Duration

```
Task 1 → 2 sec
Task 2 → 3 sec
Task 3 → 120 sec  ❗
Task 4 → 2 sec
```

👉 One task taking much longer → Skew

---

### ❗ Uneven Input Size

```
Task 1 → 10 MB
Task 2 → 12 MB
Task 3 → 900 MB ❗
Task 4 → 11 MB
```

👉 One partition overloaded

---

### ❗ Shuffle Read Size

- One task has very high shuffle read

---

## 🔹 Conclusion

👉 If ONE task is slow → Data Skew  
👉 If ALL tasks slow → Resource issue  

---

# 2️⃣ Salting vs AQE vs Broadcast (When to Use What)

---

## 🔥 Comparison Table

| Feature | Salting | AQE | Broadcast |
|----------|----------|------|------------|
| Type | Manual | Automatic | Optimization |
| Use Case | Heavy skew | Moderate skew | Small table |
| Shuffle Reduction | Partial | Yes | Yes |
| Complexity | High | Low | Low |
| Control | Full | Limited | Medium |

---

## 🔹 When to Use What

---

### ✅ Use Broadcast Join

- One table is small (<10MB)
- Want fastest performance
- Avoid shuffle

```python
df1.join(broadcast(df2), "id")
```

---

### ✅ Use AQE

- Spark 3+
- Moderate skew
- Want automatic handling

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

### ✅ Use Salting

- Heavy skew (e.g., 90% data on one key)
- AQE not enough
- Broadcast not possible

---

## 🔥 Decision Flow (Important)

```
Small table? → Broadcast
Else → Enable AQE
Still skew? → Apply Salting
```

---

# 3️⃣ Real Interview Case Study (Company-Level 🔥)

---

## 📌 Scenario

Company: E-commerce Platform

---

## 🔹 Problem

- Orders table → 500M rows
- Users table → 5M rows
- Join taking 30+ minutes

---

## 🔹 Investigation

Spark UI shows:

- One task taking 10x longer
- Key "guest_user" dominating

👉 Data skew detected

---

## 🔹 Solution Steps

---

### Step 1: Try Broadcast

```python
orders.join(broadcast(users), "user_id")
```

❌ Not possible (users table too large)

---

### Step 2: Enable AQE

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
```

👉 Improved but still slow

---

### Step 3: Apply Salting

```python
orders_salted = orders.withColumn("salt", floor(rand() * 10))
users_salted = users.withColumn("salt", explode(array([0,1,2,3,4,5,6,7,8,9])))

result = orders_salted.join(users_salted, ["user_id", "salt"])
```

---

## 🔹 Result

- Skew removed  
- Job reduced from 30 min → 8 min  
- Balanced tasks  

---

## 🔹 Interview Answer Style

👉 Problem → Analysis → Solution → Result

---

# 4️⃣ Mock Interview (Joins + Skew + Performance 🔥)

---

## 🎯 Q1: Your join is very slow, what will you check first?

👉 Check Spark UI → Stages → Tasks → Look for skew or shuffle.

---

## 🎯 Q2: How do you identify data skew?

👉 One task takes significantly longer and processes more data.

---

## 🎯 Q3: How do you fix skew?

- Broadcast join  
- AQE  
- Salting  

---

## 🎯 Q4: Why is shuffle expensive?

👉 Network + disk + serialization overhead.

---

## 🎯 Q5: When will you use broadcast join?

👉 When one dataset is small enough to fit in memory.

---

## 🎯 Q6: Difference between AQE and Salting?

- AQE → Automatic  
- Salting → Manual  

---

## 🎯 Q7: What is the biggest performance bottleneck in Spark?

👉 Shuffle and skew.

---

## 🎯 Q8: How do you reduce shuffle?

- Broadcast  
- Partitioning  
- Bucketing  

---

## 🎯 Q9: What happens if executor runs out of memory?

👉 Spill to disk or task failure.

---

## 🎯 Q10: How do you debug slow Spark job?

👉 Spark UI + execution plan + partition analysis.

---

# 🎯 Final Interview Strategy

---

## 🔥 Always Answer in This Flow

1. Identify problem  
2. Use Spark UI  
3. Explain root cause  
4. Apply optimization  
5. Show improvement  

---

# 🚀 Final Summary

```
Skew Detection → Spark UI
Optimization → Broadcast / AQE / Salting
Goal → Balanced partitions + reduced shuffle
```

---

# 🔥 Golden Rule

👉 One slow task = Skew  
👉 Many slow tasks = Resource issue  

# 🚀 Spark Caching, Persist, Hashing & Broadcast (Interview Deep Dive)

---

# 1️⃣ Caching in Spark

---

## 📌 What is Cache?

> Caching stores data in memory to reuse it without recomputation.

---

## 🔹 Syntax

```python
df.cache()
```

---

## 🔹 Example

```python
df = spark.read.parquet("/data")

df_filtered = df.filter("age > 25")

df_filtered.cache()

df_filtered.count()   # First time → computation
df_filtered.show()    # Second time → from cache
```

---

## 🔹 Why Use Cache?

Without cache:

```
Every action → recompute full DAG
```

With cache:

```
First action → compute + store
Next actions → reuse cached data
```

---

## 🔹 When to Use Cache?

- Data reused multiple times  
- Expensive transformations  
- Iterative algorithms  

---

## ❌ When NOT to Use

- Data used only once  
- Very large dataset (memory issue)  

---

# 2️⃣ Persist in Spark

---

## 📌 What is Persist?

> Persist allows storing data in different storage levels (memory, disk).

df.cache() = df.persist(StorageLevel.MEMORY_AND_DISK)

---

## 🔹 Syntax

```python
df.persist()
```

---

## 🔹 Storage Levels

```python
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_ONLY)
df.persist(StorageLevel.MEMORY_AND_DISK)
df.persist(StorageLevel.DISK_ONLY)
```
**MEMORY_ONLY**
- Data is stored in RAM as deserialized Java objects.
- If not enough memory, recompute the partitions when needed.

**MEMORY_AND_DISK**
- Tries to store in memory first.
- If memory is not enough, spill the rest to disk.

**DISK_ONLY**
- Stores data only on disk.
- Slowest option.

**MEMORY_ONLY_2**
- 2x Replicated

**OFF_HEAP (Experimental)**
- Uses off-heap memory (outside JVM heap)
- Must be enabled with spark.memory.offHeap.enabled=true.

---

## 🔹 Example

```python
df.persist(StorageLevel.MEMORY_AND_DISK)
```

---

## 🔹 Cache vs Persist

| Feature | Cache | Persist |
|----------|--------|----------|
| Default level | MEMORY_ONLY | Custom |
| Flexibility | Low | High |
| Usage | Simple | Advanced |

---

## 🔹 Interview Answer

👉 Cache is shorthand for persist with MEMORY_ONLY.

---

# 3️⃣ Hashing in Spark (Very Important 🔥)

---

## 📌 What is Hashing?

> Spark distributes data using hash function during partitioning.

---

## 🔹 Formula

```
Partition = hash(key) % num_partitions
```

---

## 🔹 Example

```python
df.repartition("id")
```

---

## 🔹 What Happens?

```
id=10 → hash(10)%4 → Partition 2
id=20 → hash(20)%4 → Partition 1
```

---

## 🔹 Where Hashing is Used?

- Joins  
- groupBy  
- reduceByKey  
- partitioning  

---

## 🔹 Why Important?

- Ensures same keys go to same partition  
- Enables correct joins and aggregations  

---

# 4️⃣ Broadcast Variable

---

## 📌 What is Broadcast Variable?

> Read-only variable shared across all executors.

---

## 🔹 Syntax

```python
broadcast_var = spark.sparkContext.broadcast([1,2,3])
```

---

## 🔹 Example

```python
data = [1,2,3]
b = spark.sparkContext.broadcast(data)

rdd.map(lambda x: x in b.value)
```

---

## 🔹 Why Use?

- Avoid sending same data multiple times  
- Reduce network overhead  

---

# 5️⃣ Broadcast Join

---

## 📌 What is Broadcast Join?

> Small dataset is broadcast to all executors to avoid shuffle.

---

## 🔹 Syntax

```python
from pyspark.sql.functions import broadcast

df1.join(broadcast(df2), "id")
```

---

## 🔹 How It Works

```
Small table → Broadcast
Large table → Remains distributed
Join → Happens locally
```

---

## 🔹 Benefits

- No shuffle for large dataset  
- Faster execution  

---

# 6️⃣ Broadcast Variable vs Broadcast Join

---

## 🔥 Comparison

| Feature | Broadcast Variable | Broadcast Join |
|----------|-------------------|----------------|
| Type | Data sharing | Join optimization |
| Use case | Small lookup data | Small table join |
| API | sparkContext.broadcast() | broadcast() function |
| Shuffle avoided | ❌ | ✅ |

---

# 7️⃣ When to Use What?

---

## 🔹 Use Cache

- Reusing same DataFrame multiple times  

---

## 🔹 Use Persist

- Large data → need disk fallback  

---

## 🔹 Use Broadcast Variable

- Small lookup/reference data  
- Used inside transformations  

---

## 🔹 Use Broadcast Join

- One table is small  
- Want to avoid shuffle  

---

# 8️⃣ Hands-On Combined Example

---

```python
from pyspark.sql.functions import broadcast
from pyspark import StorageLevel

df_large = spark.range(0, 1000000)
df_small = spark.range(0, 1000)

# Cache large dataset
df_large.cache()

# Persist with disk fallback
df_small.persist(StorageLevel.MEMORY_AND_DISK)

# Broadcast join
result = df_large.join(broadcast(df_small), "id")

result.show()
```

---

# 9️⃣ Interview Questions

---

## ❓ Difference between cache and persist?

👉 Cache = MEMORY_ONLY  
👉 Persist = Custom storage levels  

---

## ❓ When to use broadcast join?

👉 When one dataset is small  

---

## ❓ What is hashing in Spark?

👉 Method to distribute data across partitions using hash(key)

---

## ❓ Difference between broadcast variable and broadcast join?

👉 Broadcast variable → share data  
👉 Broadcast join → optimize join  

---

## ❓ What happens if cache memory is full?

👉 Data is evicted or spilled to disk (if persisted)

---

# 🎯 Interview Strategy

---

## Always Explain:

1. What it is  
2. Why needed  
3. When to use  
4. Example  
5. Performance impact  

---

# 🚀 Final Summary

```
Cache → Reuse data
Persist → Flexible storage
Hashing → Partition distribution
Broadcast Variable → Share small data
Broadcast Join → Avoid shuffle
```

---

# 🔥 Golden Rules

- Cache only reused data  
- Use broadcast for small tables  
- Avoid unnecessary persist  
- Hashing ensures correct partitioning  

## Caching Hands On

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Create first DataFrame

data1 = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Esther"),
    (6, "Fanny"),
    (7, "Gabriel"),
]

df1 = spark.createDataFrame(data1, ["id", "name"])

In [0]:
df1.display()

In [0]:
df1 = df1.withColumn("Flag", lit("Yes"))

In [0]:
display(df1)

In [0]:
# Applying Caching to df1
df1.cache()

In [0]:
df2 = df1.filter(col('id')== 1)

In [0]:
display(df2)

In [0]:
df2.explain()

## Persist Hands On

In [0]:
from pyspark.storagelevel import StorageLevel

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.persist(StorageLevel.MEMORY_ONLY)

In [0]:
df2 = df1.filter(col('id')== 1)
df2.display()

In [0]:
df2.explain()

## For Unpersisting Cache & Persist

In [0]:
df1.unpersist()

## Pivot vs Unpivot

In [0]:
# df_unpivot = df.unpivot(
#     ids=["id"],
#     values=["jan", "feb", "mar"],
#     variableColumnName="month",
#     valueColumnName="value"
# )

# 🚀 Spark Client Mode vs Cluster Mode (Deep Dive)

---

# 1️⃣ What is Deployment Mode in Spark?

## 📌 Definition

> Deployment mode defines **where the Driver program runs** in a Spark application.

---

## 🔹 Two Modes

- Client Mode  
- Cluster Mode  

---

# 2️⃣ Client Mode

---

## 📌 Definition

> In client mode, the **Driver runs on the client machine** (your laptop / Databricks notebook).

---

## 🔹 Architecture

```
Client Machine (Driver)
        ↓
Cluster Manager (YARN / Kubernetes / Databricks)
        ↓
Executors (Worker Nodes)
```

---

## 🔹 Flow

1. User submits job  
2. Driver runs on client  
3. Executors run on cluster  
4. Driver communicates with executors  

---

## 🔹 Example

```bash
spark-submit --deploy-mode client app.py
```

---

## 🔹 Key Points

- Driver outside cluster  
- Needs stable network connection  
- Good for development & debugging  

---

## 🔹 Pros

- Easy debugging  
- Direct logs  
- Interactive usage  

---

## 🔹 Cons

- Network dependency  
- Driver failure risk  
- Not ideal for production  

---

# 3️⃣ Cluster Mode

---

## 📌 Definition

> In cluster mode, the **Driver runs inside the cluster**.

---

## 🔹 Architecture

```
Client Machine (Submit Request)
        ↓
Cluster Manager
        ↓
Driver (inside cluster)
        ↓
Executors
```

---

## 🔹 Flow

1. User submits job  
2. Cluster manager launches driver  
3. Driver runs inside cluster  
4. Executors execute tasks  

---

## 🔹 Example

```bash
spark-submit --deploy-mode cluster app.py
```

---

## 🔹 Key Points

- Driver inside cluster  
- No dependency on client  
- Suitable for production  

---

## 🔹 Pros

- More reliable  
- Fault-tolerant  
- Better for long-running jobs  

---

## 🔹 Cons

- Harder debugging  
- Logs not directly visible  

---

# 4️⃣ Client vs Cluster Mode (Comparison)

---

| Feature | Client Mode | Cluster Mode |
|----------|-------------|---------------|
| Driver Location | Client machine | Cluster |
| Debugging | Easy | Hard |
| Network dependency | High | Low |
| Production use | ❌ | ✅ |
| Failure impact | High | Low |

---

# 5️⃣ Real-World Usage

---

## 🔹 Use Client Mode When

- Development  
- Testing  
- Notebooks (Databricks default)  

---

## 🔹 Use Cluster Mode When

- Production pipelines  
- Long-running jobs  
- Batch processing  

---

# 6️⃣ Databricks Perspective

---

## 🔹 In Databricks

- Driver runs inside cluster  
- Behaves like cluster mode  

---

## 🔹 Notebook Execution

- Driver = Cluster driver node  
- Executors = Worker nodes  

---

# 7️⃣ Interview Questions

---

## ❓ Difference between client and cluster mode?

👉 Client → Driver on client  
👉 Cluster → Driver inside cluster  

---

## ❓ Which mode is better for production?

👉 Cluster mode  

---

## ❓ Why client mode is risky?

👉 Driver depends on local machine/network  

---

## ❓ What happens if driver fails?

👉 Job fails  

---

## ❓ Where does driver run in Databricks?

👉 Inside cluster  

---

# 🎯 Interview Answer (Best)

Client mode runs the driver on the client machine, while cluster mode runs the driver inside the cluster. Cluster mode is preferred for production as it is more reliable and fault-tolerant.

---

# 🚀 Final Summary

```
Client Mode → Driver on local machine
Cluster Mode → Driver inside cluster
```

---

# 🔥 Golden Rule

👉 Development → Client Mode  
👉 Production → Cluster Mode  

# 🚀 Adaptive Query Execution (AQE) in Spark (Deep Dive)

---

# 1️⃣ What is AQE?

## 📌 Definition

> Adaptive Query Execution (AQE) is a Spark optimization feature that dynamically changes the execution plan at runtime based on actual data statistics.

---

## 🔹 Key Idea

Instead of:

```
Static Plan (before execution)
```

Spark does:

```
Dynamic Plan (optimized during execution)
```

---

# 2️⃣ Why AQE is Needed?

---

## 🔥 Problem Without AQE

- Spark creates execution plan before running  
- Does not know actual data size  
- Cannot handle skew properly  

---

## 🔹 Example

```python
df1.join(df2, "id")
```

Spark assumes:

- Both tables large → Sort Merge Join  

But actual:

- One table is small  

👉 Missed optimization ❌  

---

## 🔥 With AQE

- Detects small table at runtime  
- Switches to Broadcast Join  

👉 Faster execution ✅  

---

# 3️⃣ How AQE Works

---

## 🔹 Execution Flow

```
1. Initial Plan Created
2. Job Starts Execution
3. Spark Collects Runtime Statistics
4. Plan is Re-Optimized
5. Optimized Plan Executed
```

---

# 4️⃣ AQE Features (VERY IMPORTANT 🔥)

---

# 🔹 1. Dynamic Join Selection

---

## 📌 What It Does

Changes join type at runtime.

---

## 🔹 Example

Before:

```
SortMergeJoin
```

After AQE:

```
BroadcastHashJoin
```

---

# 🔹 2. Skew Join Handling

---

## 📌 What It Does

- Detects skewed partitions  
- Splits large partitions  

---

## 🔹 Benefit

- Avoids long-running tasks  
- Balances workload  

---

# 🔹 3. Coalesce Shuffle Partitions

---

## 📌 What It Does

- Reduces number of partitions dynamically  

---

## 🔹 Example

Before:

```
200 partitions
```

After AQE:

```
20 partitions
```

---

## 🔹 Benefit

- Less overhead  
- Faster execution  

---

# 🔹 4. Dynamic Partition Pruning

---

## 📌 What It Does

- Filters partitions dynamically during join  

---

## 🔹 Benefit

- Reads less data  
- Improves performance  

---

# 5️⃣ Enable AQE

---

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
```

---

## 🔹 Enable Skew Handling

```python
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

# 6️⃣ Hands-On Example

---

```python
df1 = spark.range(0, 1000000)
df2 = spark.range(0, 1000)

# Without AQE
df1.join(df2, "id").explain()

# Enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

df1.join(df2, "id").explain()
```

---

## 🔹 Observe

- Join type changes  
- Number of partitions reduced  

---

# 7️⃣ AQE vs Static Execution

---

| Feature | Without AQE | With AQE |
|----------|-------------|-----------|
| Plan | Static | Dynamic |
| Join Optimization | No | Yes |
| Skew Handling | Manual | Automatic |
| Partition Tuning | Fixed | Dynamic |

---

# 8️⃣ When AQE Helps Most

---

- Joins  
- Skewed data  
- Large shuffle operations  
- Unknown data distribution  

---

# 9️⃣ Limitations

---

- Slight overhead for optimization  
- Not useful for very small datasets  

---

# 🔟 Interview Questions

---

## ❓ What is AQE?

👉 Runtime optimization of execution plan.

---

## ❓ How does AQE improve performance?

👉 Adjusts join type, partitions, and skew handling dynamically.

---

## ❓ What are AQE features?

- Dynamic join selection  
- Skew handling  
- Partition coalescing  
- Partition pruning  

---

## ❓ Difference between AQE and Catalyst?

- Catalyst → Compile-time optimization  
- AQE → Runtime optimization  

---

# 🎯 Interview Answer (Best)

Adaptive Query Execution (AQE) is a Spark optimization feature that dynamically adjusts the execution plan at runtime based on actual data statistics to improve performance by optimizing joins, handling skew, and reducing shuffle partitions.

---

# 🚀 Final Summary

```
Static Plan → Limited optimization
AQE → Dynamic optimization at runtime
```

---

# 🔥 Golden Rule

👉 Enable AQE for production workloads 🚀  

# 🚀 Window Functions in Spark (Deep Dive for Interviews)

---

# 1️⃣ What is a Window Function?

## 📌 Definition

> A window function performs calculations across a set of rows related to the current row without collapsing the dataset.

---

## 🔹 Key Idea

Unlike `groupBy()`:

- groupBy → Aggregates → Reduces rows  
- Window → Keeps all rows → Adds new column  

---

## 🔹 Example

```python
df.groupBy("dept").avg("salary")
```

👉 Returns 1 row per dept  

---

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

df.withColumn("avg_salary", avg("salary").over(Window.partitionBy("dept")))
```

👉 Keeps all rows + adds avg column  

---

# 2️⃣ Window Specification

---

## 🔹 Syntax

```python
Window.partitionBy().orderBy().rowsBetween()
```

---

## 🔹 Components

### 1. partitionBy()

- Groups data (like groupBy)
- Defines window

---

### 2. orderBy()

- Defines order within partition

---

### 3. rowsBetween()

- Defines frame (range of rows)

---

# 3️⃣ Types of Window Functions

---

# 🔹 1. Ranking Functions

---

## Example Data

```python
data = [
    ("A", 100),
    ("A", 200),
    ("A", 200),
    ("B", 300)
]

df = spark.createDataFrame(data, ["dept", "salary"])
```

---

## 🔹 ROW_NUMBER

```python
from pyspark.sql.functions import row_number

window = Window.partitionBy("dept").orderBy("salary")

df.withColumn("row_num", row_number().over(window))
```

👉 Unique ranking (no duplicates)

---

## 🔹 RANK

```python
from pyspark.sql.functions import rank

df.withColumn("rank", rank().over(window))
```

👉 Same rank for duplicates (gap exists)

---

## 🔹 DENSE_RANK

```python
from pyspark.sql.functions import dense_rank

df.withColumn("dense_rank", dense_rank().over(window))
```

👉 No gaps in ranking

---

# 🔥 Difference

| Function | Duplicate Handling | Gaps |
|----------|-------------------|------|
| row_number | Unique | No |
| rank | Same rank | Yes |
| dense_rank | Same rank | No |

---

# 🔹 2. Aggregate Window Functions

---

## Example

```python
from pyspark.sql.functions import sum, avg

window = Window.partitionBy("dept")

df.withColumn("total_salary", sum("salary").over(window))
df.withColumn("avg_salary", avg("salary").over(window))
```

---

# 🔹 3. Analytical Functions

---

## 🔹 LAG

```python
from pyspark.sql.functions import lag

window = Window.orderBy("salary")

df.withColumn("prev_salary", lag("salary", 1).over(window))
```

---

## 🔹 LEAD

```python
from pyspark.sql.functions import lead

df.withColumn("next_salary", lead("salary", 1).over(window))
```

---

## 🔹 Use Case

- Time series analysis  
- Comparing previous/next row  

---

# 🔹 4. Frame-Based Functions

---

## Example

```python
window = Window.orderBy("salary").rowsBetween(-1, 1)

df.withColumn("moving_avg", avg("salary").over(window))
```

---

## 🔹 Meaning

```
Current row ± 1 row
```

---

# 4️⃣ Real-World Use Cases

---

## 🔹 1. Deduplication

```python
window = Window.partitionBy("id").orderBy("timestamp")

df.withColumn("rn", row_number().over(window)) \
  .filter("rn = 1")
```

---

## 🔹 2. Top N per Group

```python
window = Window.partitionBy("dept").orderBy(col("salary").desc())

df.withColumn("rank", row_number().over(window)) \
  .filter("rank <= 3")
```

---

## 🔹 3. Running Total

```python
window = Window.partitionBy("dept").orderBy("salary")

df.withColumn("running_total", sum("salary").over(window))
```

---

# 5️⃣ Performance Considerations

---

## 🔥 Window Functions Cause Shuffle

- partitionBy → Shuffle  
- orderBy → Sorting  

---

## 🔹 Optimization Tips

- Reduce partitions  
- Use proper partition keys  
- Avoid unnecessary orderBy  
- Use broadcast if joining before window  

---

# 6️⃣ Window vs GroupBy

---

| Feature | Window | GroupBy |
|----------|---------|----------|
| Output rows | Same | Reduced |
| Shuffle | Yes | Yes |
| Use case | Analytics | Aggregation |

---

# 7️⃣ Interview Questions

---

## ❓ What is window function?

👉 Performs calculation across rows without reducing dataset.

---

## ❓ Difference between rank and dense_rank?

👉 rank has gaps, dense_rank does not.

---

## ❓ When to use window instead of groupBy?

👉 When you need row-level detail.

---

## ❓ Why window is expensive?

👉 Because of shuffle and sorting.

---

## ❓ What is lag function?

👉 Gets previous row value.

---

# 🎯 Interview Answer (Best)

Window functions allow performing calculations across a partition of data while retaining individual rows, commonly used for ranking, running totals, and analytical comparisons.

---

# 🚀 Final Summary

```
Window Function:
- Partition data
- Order rows
- Apply function
- Keep all rows
```

---

# 🔥 Golden Rules

- Use window for row-level analytics  
- Avoid unnecessary sorting  
- Optimize partitioning  
- Watch shuffle in Spark UI  

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window = Window.orderBy(col("salary").desc())

df_rank = df_employee.withColumn("rnk", dense_rank().over(window))

df_result = df_rank.filter(col("rnk") == 2)\
    .select(col("salary").alise("second_highest_salary"))\
    .distinct()

df_result.show()

# 🚀 Delta Lake & Streaming (Event Hub + Spark) – Deep Dive

---

# 🔥 PART 1: Delta Lake Deep Dive

---

# 1️⃣ What is Delta Lake?

## 📌 Definition

> Delta Lake is a storage layer on top of data lakes that provides ACID transactions, schema enforcement, and versioning.

---

## 🔹 Why Delta Lake?

Problems in traditional data lake:

- No ACID guarantees  
- Data corruption risk  
- No versioning  
- Slow queries  

---

## 🔹 Delta Lake Solves

- ACID transactions  
- Time travel  
- Schema enforcement  
- Faster reads  

---

# 2️⃣ ACID Transactions (Very Important 🔥)

---

## 📌 ACID Meaning

- **A** → Atomicity  
- **C** → Consistency  
- **I** → Isolation  
- **D** → Durability  

---

## 🔹 Example

```python
df.write.format("delta").mode("overwrite").save("/delta/table")
```

---

## 🔹 How Delta Achieves ACID?

👉 Uses **Transaction Log (_delta_log)**

---

## 🔹 Transaction Log

```
/delta/table/
   ├── data files
   └── _delta_log/
         ├── 000000.json
         ├── 000001.json
```

---

## 🔹 What It Stores?

- Metadata  
- Schema  
- File changes  

---

# 3️⃣ Time Travel

---

## 📌 What is Time Travel?

> Ability to query previous versions of data.

---

## 🔹 Example

```python
spark.read.format("delta") \
  .option("versionAsOf", 0) \
  .load("/delta/table")
```

---

## 🔹 Use Cases

- Audit  
- Rollback  
- Debugging  

---

# 4️⃣ OPTIMIZE (File Compaction)

---

## 📌 Problem

Small files problem:

```
1000 small files → slow queries
```

---

## 🔹 Solution

```sql
OPTIMIZE delta.`/delta/table`
```

---

## 🔹 What It Does

- Merges small files into larger files  
- Improves read performance  

---

# 5️⃣ ZORDER (Very Important 🔥)

---

## 📌 What is ZORDER?

> Co-locates related data in same files for faster filtering.

---

## 🔹 Example

```sql
OPTIMIZE delta.`/delta/table`
ZORDER BY (user_id)
```

---

## 🔹 Benefit

Query:

```sql
SELECT * FROM table WHERE user_id = 100
```

👉 Reads fewer files → faster  

---

# 🔥 OPTIMIZE vs ZORDER

| Feature | OPTIMIZE | ZORDER |
|----------|-----------|----------|
| Purpose | File compaction | Data clustering |
| Improves | File size | Query speed |
| Usage | Always | Filter-heavy queries |

---

# 6️⃣ Delta Lake Interview Questions

---

## ❓ How Delta ensures ACID?

👉 Using transaction log (_delta_log)

---

## ❓ What is time travel?

👉 Accessing previous versions of data

---

## ❓ Why OPTIMIZE?

👉 To reduce small files problem

---

## ❓ What is ZORDER?

👉 Improves query performance by clustering data

---

---

# 🔥 PART 2: Streaming (Event Hub + Spark)

---

# 1️⃣ What is Structured Streaming?

---

## 📌 Definition

> Spark Structured Streaming processes real-time data using DataFrame API.

---

## 🔹 Key Concept

```
Stream = Infinite Data
```

---

# 2️⃣ Event Hub + Spark Architecture

---

## 🔹 Flow

```
Event Hub → Spark Streaming → Processing → Sink (Delta Lake)
```

---

# 3️⃣ Reading from Event Hub

---

## 🔹 Example

```python
df = spark.readStream \
  .format("eventhubs") \
  .options(**ehConf) \
  .load()
```

---

# 4️⃣ Streaming Transformation

---

```python
df_parsed = df.selectExpr("CAST(body AS STRING)")
```

---

# 5️⃣ Writing Stream to Delta

---

```python
df_parsed.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/checkpoint") \
  .start("/delta/output")
```

---

# 🔥 Important: Checkpointing

---

## 📌 What is Checkpoint?

> Stores streaming state for fault tolerance.

---

## 🔹 Why Needed?

- Recover from failure  
- Maintain offsets  
- Ensure exactly-once processing  

---

# 6️⃣ Output Modes

---

| Mode | Description |
|------|-------------|
| append | New rows only |
| complete | Entire table |
| update | Changed rows |

---

# 7️⃣ Windowing in Streaming

---

## 🔹 Example

```python
from pyspark.sql.functions import window

df.groupBy(window("timestamp", "10 minutes")).count()
```

---

# 8️⃣ Watermarking

---

## 📌 What is Watermark?

> Handles late-arriving data.

---

## 🔹 Example

```python
df.withWatermark("timestamp", "10 minutes")
```

---

# 9️⃣ Streaming Interview Questions

---

## ❓ What is checkpointing?

👉 Stores state for fault tolerance.

---

## ❓ What is watermark?

👉 Handles late data in streaming.

---

## ❓ Difference between batch and streaming?

- Batch → finite  
- Streaming → continuous  

---

## ❓ What happens if streaming job fails?

👉 Restarts from checkpoint  

---

# 🔟 Real-World Use Case

---

## Scenario

- IoT sensors send data → Event Hub  
- Spark processes data  
- Store in Delta Lake  

---

## Benefits

- Real-time analytics  
- Fault tolerance  
- Scalable  

---

# 🎯 Final Summary

---

## Delta Lake

- ACID transactions  
- Time travel  
- OPTIMIZE  
- ZORDER  

---

## Streaming

- Event Hub ingestion  
- Structured Streaming  
- Checkpointing  
- Watermarking  

---

# 🔥 Golden Rules

- Use Delta for reliable storage  
- Always use checkpoint in streaming  
- Use OPTIMIZE for performance  
- Use ZORDER for filtering queries  

# 🚀 SQL Normalization (Deep Dive for Interviews)

---

# 1️⃣ What is Normalization?

## 📌 Definition

> Normalization is the process of organizing data in a database to reduce redundancy and improve data integrity.

---

## 🔹 Goals

- Remove duplicate data  
- Ensure consistency  
- Improve data integrity  
- Avoid anomalies  

---

# 2️⃣ Problems Without Normalization

---

## 🔥 Example (Unnormalized Table)

| student_id | student_name | course | instructor |
|------------|--------------|--------|------------|
| 1 | Arijit | SQL | John |
| 1 | Arijit | Python | Mike |
| 2 | Ravi | SQL | John |

---

## 🔹 Issues

### ❌ Redundancy

- "Arijit" repeated multiple times  

---

### ❌ Update Anomaly

- Change name → must update multiple rows  

---

### ❌ Insert Anomaly

- Cannot insert student without course  

---

### ❌ Delete Anomaly

- Deleting last course → student info lost  

---

# 3️⃣ Normal Forms

---

# 🔹 1NF (First Normal Form)

---

## 📌 Rule

- No repeating groups  
- Atomic values (no multiple values in a column)

---

## ❌ Not in 1NF

| id | courses |
|----|----------|
| 1 | SQL, Python |

---

## ✅ Convert to 1NF

| id | course |
|----|--------|
| 1 | SQL |
| 1 | Python |

---

---

# 🔹 2NF (Second Normal Form)

---

## 📌 Rule

- Must be in 1NF  
- No partial dependency  

---

## 🔹 Partial Dependency

When a column depends on part of composite key.

---

## ❌ Example

| student_id | course | student_name |
|------------|--------|--------------|

Primary Key = (student_id, course)

Problem:

- student_name depends only on student_id  

---

## ✅ Solution

Split into:

### Student Table

| student_id | student_name |

---

### Course Table

| student_id | course |

---

---

# 🔹 3NF (Third Normal Form)

---

## 📌 Rule

- Must be in 2NF  
- No transitive dependency  

---

## 🔹 Transitive Dependency

A → B → C  
(Non-key depends on another non-key)

---

## ❌ Example

| student_id | dept_id | dept_name |

Problem:

- dept_name depends on dept_id (not directly on student_id)

---

## ✅ Solution

### Student Table

| student_id | dept_id |

---

### Department Table

| dept_id | dept_name |

---

---

# 🔹 BCNF (Boyce-Codd Normal Form)

---

## 📌 Rule

- Every determinant must be a candidate key  

---

## 🔹 Example

| teacher | subject | room |

Problem:

- teacher → subject  
- subject → room  

But subject is not a key  

---

## ✅ Solution

Split into:

- Teacher → Subject  
- Subject → Room  

---

---

# 4️⃣ Summary of Normal Forms

---

| Normal Form | Rule |
|--------------|------|
| 1NF | Atomic values |
| 2NF | No partial dependency |
| 3NF | No transitive dependency |
| BCNF | Stronger version of 3NF |

---

# 5️⃣ Real-World Example

---

## Unnormalized

| order_id | customer_name | product | price |

---

## Normalized

### Customers

| customer_id | name |

---

### Orders

| order_id | customer_id |

---

### Products

| product_id | price |

---

### Order_Items

| order_id | product_id |

---

---

# 6️⃣ Advantages of Normalization

---

- Reduces redundancy  
- Improves consistency  
- Avoids anomalies  
- Efficient updates  

---

# 7️⃣ Disadvantages

---

- More joins required  
- Slightly slower queries  
- Complex design  

---

# 8️⃣ Normalization vs Denormalization

---

| Feature | Normalization | Denormalization |
|----------|---------------|------------------|
| Redundancy | Low | High |
| Performance | Slower | Faster |
| Use case | OLTP | OLAP |

---

---

# 9️⃣ Interview Questions

---

## ❓ What is normalization?

👉 Process of organizing data to reduce redundancy.

---

## ❓ Difference between 2NF and 3NF?

- 2NF → removes partial dependency  
- 3NF → removes transitive dependency  

---

## ❓ What is transitive dependency?

👉 Non-key column depends on another non-key column.

---

## ❓ When to denormalize?

👉 For performance in analytics systems.

---

## ❓ Why normalization is important?

👉 Prevents anomalies and ensures data integrity.

---

# 🎯 Interview Answer (Best)

Normalization is a database design technique used to eliminate redundancy and ensure data integrity by organizing tables into well-structured forms such as 1NF, 2NF, and 3NF.

---

# 🚀 Final Summary

```
1NF → Atomic values
2NF → Remove partial dependency
3NF → Remove transitive dependency
BCNF → Stronger 3NF
```

---

# 🔥 Golden Rule

👉 Normalize for OLTP  
👉 Denormalize for OLAP (Data Warehousing)  

# 🚀 ACID Properties in Databases (Deep Dive for Interviews)

---

# 1️⃣ What is ACID?

## 📌 Definition

> ACID is a set of properties that guarantee reliable and consistent database transactions.

---

## 🔹 ACID Stands For

- **A** → Atomicity  
- **C** → Consistency  
- **I** → Isolation  
- **D** → Durability  

---

# 2️⃣ Real-Life Example (Bank Transaction)

---

## Scenario

Transfer ₹100 from Account A → Account B

---

## Steps

1. Deduct ₹100 from A  
2. Add ₹100 to B  

---

👉 This must happen **completely or not at all**

---

# 3️⃣ Atomicity

---

## 📌 Definition

> A transaction is either fully completed or fully rolled back.

---

## 🔹 Example

```sql
BEGIN TRANSACTION;

UPDATE accounts SET balance = balance - 100 WHERE id = 'A';
UPDATE accounts SET balance = balance + 100 WHERE id = 'B';

COMMIT;
```

---

## 🔹 What If Failure Occurs?

- System crashes after deducting from A  
- B is not credited  

👉 Atomicity ensures:

```
Rollback → A gets original balance
```

---

## 🔹 Interview Answer

Atomicity ensures that all operations in a transaction are completed successfully, otherwise the entire transaction is rolled back.

---

# 4️⃣ Consistency

---

## 📌 Definition

> A transaction brings the database from one valid state to another valid state.

---

## 🔹 Example

Constraint:

```
Balance ≥ 0
```

---

## 🔹 Invalid Operation

```sql
UPDATE accounts SET balance = -500 WHERE id = 'A';
```

👉 Violates constraint  

👉 Transaction fails  

---

## 🔹 Interview Answer

Consistency ensures that database rules, constraints, and integrity are maintained before and after a transaction.

---

# 5️⃣ Isolation

---

## 📌 Definition

> Transactions execute independently without interfering with each other.

---

## 🔹 Problem Without Isolation

Two users accessing same account:

```
User1 reads balance = 1000
User2 reads balance = 1000
```

Both withdraw ₹500

Final balance = 500 ❌ (should be 0)

---

## 🔹 Isolation Levels

| Level | Description |
|------|-------------|
| Read Uncommitted | Dirty reads allowed |
| Read Committed | Only committed data |
| Repeatable Read | Same result within transaction |
| Serializable | Full isolation |

---

## 🔹 Interview Answer

Isolation ensures that concurrent transactions do not interfere with each other.

---

# 6️⃣ Durability

---

## 📌 Definition

> Once a transaction is committed, it is permanently stored.

---

## 🔹 Example

```sql
COMMIT;
```

After commit:

- Data saved to disk  
- Survives system crash  

---

## 🔹 How Achieved?

- Write-Ahead Logging (WAL)  
- Transaction logs  

---

## 🔹 Interview Answer

Durability ensures that committed transactions are permanently stored and cannot be lost.

---

# 7️⃣ ACID in Delta Lake (Very Important 🔥)

---

## 🔹 How Delta Ensures ACID?

👉 Using **Transaction Log (_delta_log)**

---

## 🔹 Example

```python
df.write.format("delta").mode("append").save("/delta/table")
```

---

## 🔹 What Happens?

- Write operation logged  
- Metadata updated  
- Data committed atomically  

---

## 🔹 Benefits

- No data corruption  
- Safe concurrent writes  
- Reliable streaming  

---

# 8️⃣ ACID vs Non-ACID Systems

---

| Feature | ACID (RDBMS/Delta) | Non-ACID (Raw Data Lake) |
|----------|-------------------|--------------------------|
| Transactions | Supported | Not supported |
| Consistency | Strong | Weak |
| Reliability | High | Low |

---

# 9️⃣ Interview Questions

---

## ❓ What is ACID?

👉 Set of properties ensuring reliable transactions.

---

## ❓ Explain Atomicity with example?

👉 Transaction either completes fully or rolls back.

---

## ❓ What is isolation?

👉 Prevents concurrent transaction conflicts.

---

## ❓ What ensures durability?

👉 Logs and persistent storage.

---

## ❓ How Delta Lake ensures ACID?

👉 Using transaction log (_delta_log).

---

# 🎯 Interview Answer (Best)

ACID properties ensure reliable and consistent database transactions by guaranteeing atomicity, consistency, isolation, and durability, which are essential for maintaining data integrity in systems like relational databases and Delta Lake.

---

# 🚀 Final Summary

```
Atomicity → All or nothing
Consistency → Valid state
Isolation → No interference
Durability → Permanent storage
```

---

# 🔥 Golden Rule

👉 ACID = Reliability + Consistency + Safety in transactions  